In [1]:
!pip install transformers accelerate bitsandbytes peft trl -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd 
from kaggle_secrets import UserSecretsClient 

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

print("HF_TOKEN is loaded : ", HF_TOKEN is not None)

ModuleNotFoundError: No module named 'kaggle_secrets'

In [ ]:
from huggingface_hub import login 
import pandas as pd 

login(token = HF_TOKEN)

df = pd.read_csv('/kaggle/input/datasets/pravargarg123/anemia-detection-medgamma/master.csv', keep_default_na = False)

train_df = df[df['split'] == 'train'].reset_index(drop = True)
val_df = df[df['split'] == 'valid'].reset_index(drop = True)

print('The shape of the training split :', train_df.shape)
print('The shape of the validation split : ', val_df.shape)
print(df['severity'].value_counts())

The shape of the training split : (66, 5)
The shape of the validation split :  (14, 5)
severity
None        39
Moderate    38
Mild        16
Severe       2
Name: count, dtype: int64


In [ ]:
# Loading the MedGamma model from HuggingFace

from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_4bit = True)

model_id = 'google/medgemma-4b-it'

processor = AutoProcessor.from_pretrained(model_id, token = HF_TOKEN)

model = AutoModelForImageTextToText.from_pretrained(model_id, quantization_config = bnb_config, device_map = 'auto', token = HF_TOKEN)


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model 

lora_config = LoraConfig(r = 16, target_modules = ["q_proj", "v_proj"], lora_alpha = 32, lora_dropout = 0.05, task_type = "CAUSAL_LM")

lora_model = get_peft_model(model,lora_config)

lora_model.print_trainable_parameters()

trainable params: 6,447,104 || all params: 4,306,526,576 || trainable%: 0.1497


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
from datasets import Datasets 